# Getting Started with Biomedical Knowledge Lookup

This notebook shows you how to get started with the Biomedical Knowledge Lookup library.

## Installation

First, install the library:

In [ ]:
# Using pip
# pip install biomedical-knowledge-lookup

# Using Poetry
# poetry add biomedical-knowledge-lookup

# From source (development)
# poetry install

## Basic Usage

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup, KnowledgeSource

# Initialize the lookup engine
lookup = CentralKnowledgeLookup()

# Search for diabetes across multiple sources
result = await lookup.search_concepts(
    query="diabetes",
    sources=[
        KnowledgeSource.MONDO,
        KnowledgeSource.OPENTARGETS,
        KnowledgeSource.DISGENET
    ]
)

# Display results
print(f"Found {len(result.concepts)} concepts")
for concept in result.concepts[:5]:
    print(f"\n{concept.primary_label}:")
    print(f"  ID: {concept.primary_id}")
    print(f"  Type: {concept.concept_type.value}")

# Clean up
await lookup.close()

## Search by Entity Type

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup, KnowledgeSource

lookup = CentralKnowledgeLookup()

# Search for a specific gene
results = await lookup.search_concepts(
    query="BRCA1",
    sources=[KnowledgeSource.OPENTARGETS]
)

print(f"Found {len(results.concepts)} gene concepts for BRCA1")
for concept in results.concepts:
    print(f"  {concept.primary_id} - {concept.primary_label}")

await lookup.close()

## Get Concept Details

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup, KnowledgeSource

lookup = CentralKnowledgeLookup()

# Get detailed information for a specific concept
details = await lookup.get_concept_details(
    concept_id="MONDO:0005148",  # Diabetes mellitus
    sources=[KnowledgeSource.MONDO]
)

if details:
    print(f"Concept: {details.primary_label}")
    print(f"ID: {details.primary_id}")
    print(f"Type: {details.concept_type.value}")
    print(f"Description: {details.description}")
    print(f"Synonyms: {', '.join(details.synonyms[:5])}")
else:
    print("Concept not found")

await lookup.close()

## Configuration

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup, LookupConfig

# Configure with custom settings
config = LookupConfig(
    cache_enabled=True,           # Enable caching
    cache_ttl=3600,              # Cache for 1 hour
    timeout_per_source=30.0,     # 30 second timeout
    max_results_per_source=50    # Max 50 results per source
)

lookup = CentralKnowledgeLookup(config)

# Search with configuration
result = await lookup.search_concepts(
    query="diabetes",
    sources=[KnowledgeSource.MONDO]
)

print(f"Results: {len(result.concepts)} concepts")

await lookup.close()

## UMLS Advanced Features

The UMLS adapter provides six advanced features beyond basic search. These require an API key set in `UMLS_API_KEY` or `UMLS_API_KEY_TU`.

Features demonstrated below:
1. **Source-restricted search** — only return concepts from a specific vocabulary (e.g. SNOMEDCT_US)
2. **Semantic-type filter** — filter results by UMLS semantic type TUI (e.g. T047 = Disease)
3. **Bulk search** — multiple queries in one API call
4. **Mappings** — find source-specific identifiers for a UMLS CUI
5. **Relationships** — retrieve parent/child/sibling relations
6. **Streaming iterators** — paginated access to definitions and relations

In [ ]:
import asyncio, os
from knowledge_lookup import LookupConfig, create_knowledge_lookup
from knowledge_lookup.models import KnowledgeSource

# UMLS requires an API key
api_key = os.environ.get("UMLS_API_KEY", "")
config = LookupConfig(api_keys={"umls": api_key})
lookup = create_knowledge_lookup(enabled_sources=[KnowledgeSource.UMLS], api_keys=config.api_keys)

if lookup:
    adapter = lookup.adapters.get(KnowledgeSource.UMLS)

    # 1. Source-restricted search
    print("--- Source-restricted search (SNOMEDCT_US) ---")
    results = await adapter.search_concepts("hypertension", limit=3, sabs="SNOMEDCT_US")
    for c in results:
        print(f"  {c.primary_id}: {c.primary_label}")

    # 2. Semantic-type filter (T047 = Disease)
    print("\n--- Semantic-type filter (T047 = Disease) ---")
    results = await adapter.search_concepts("diabetes", limit=3, semantic_types="T047")
    for c in results:
        print(f"  {c.primary_id}: {c.primary_label}  [{c.concept_type.value}]")

    # 3. Bulk search
    print("\n--- Bulk search ---")
    bulk = await adapter.bulk_search(["metformin", "atorvastatin"], limit=2)
    for q, hits in bulk.items():
        print(f"  '{q}' \u2192 {[h.primary_label for h in hits]}")

    # 4. Mappings (CUI \u2192 source IDs)
    print("\n--- Mappings: C0025598 (metformin) \u2192 RXNORM ---")
    mappings = await adapter.get_mappings("C0025598", target_source="RXNORM")
    for m in mappings[:3]:
        print(f"  {m['source']}: {m['source_id']}  \u2014  {m.get('source_name', '')}")

    # 5. Relationships
    print("\n--- Parent relations for metformin ---")
    rels = await adapter.get_relationships("C0025598", relation_labels="PAR", limit=3)
    for r in rels:
        print(f"  PAR \u2192 {r.get('related_name', '?'):40s}  ({r['related_id']})")

    # 6. Streaming
    print("\n--- Streaming iterators ---")
    def_count = sum(1 for _ in await _collect(adapter.iter_definitions("C0025598")))
    rel_count = sum(1 for _ in await _collect(adapter.iter_relations("C0025598")))
    print(f"  Metformin definitions: {def_count}")
    print(f"  Metformin relations:   {rel_count}")
else:
    print("UMLS adapter unavailable. Set UMLS_API_KEY environment variable.")

await lookup.close()
print("\nDone.")

# Helper to collect an async generator into a list
async def _collect(agen):
    return [item async for item in agen]

## Next Steps

- Check out `02-api-keys.ipynb` for API key configuration
- Check out `03-advanced.ipynb` for advanced patterns
- See `docs/examples/use_cases.md` for real-world examples